In [60]:
# Algorithms Project 1 - RSA
# Objective: implement RSA Encryption and apply it to digital signature
import pandas as pd
import numpy as np

In [61]:
# this is the library you need
import hashlib
print(hashlib.algorithms_guaranteed)

{'shake_128', 'sha224', 'blake2s', 'blake2b', 'shake_256', 'sha3_512', 'sha3_256', 'sha256', 'sha1', 'md5', 'sha3_384', 'sha512', 'sha384', 'sha3_224'}


In [62]:
h = hashlib.sha256(b'computer science at UA is the best')
m = h.hexdigest()
print(m)

e697db5d9a543ecf04f2cc76fe105eb9d65edb08e06e2ef53788157bb05dc7e6


In [63]:
# check if p is prime (most likely a prime)
import random

def FermatPrimalityTest(p, k = 5):
    ''' 
    Fermat primality test to check if a number is prime.

    Parameters:
    p: The number (int) to test for primality.
    k: The number (int) of test iterations (arbitrarily chosen).

    Returns:
    bool: True if p is likely prime or False if composite.
    '''
    # handle small numbers
    if p <= 1:
        return False
    if p == 2 or p == 3:
        return True
    if p % 2 == 0:
        return False
        
    # perform Fermat test k number of times.
    for i in range(k):
        # chose random a in the range [2, p - 2].
        a = random.randint(2, p - 2)
        # check if a^(p-1) = 1 mod p using modular exponentiation.
        if pow(a, p - 1, p) != 1:
            return False
    return True

In [64]:
# you need to modify this function to generate the two pairs of keys!!!
from math import gcd

# generate a random prime of given bit length.
def generate_prime(bits):
    '''
    Generate a random prime number.

    Parameters: 
    bits: (int) The desired bit length (b/w 128 and 256).

    Returns:
    int: A (likely) prime number.
    '''
    while True:
        # generate random odd number with specified bits
        c = random.getrandbits(bits)
        # set lowest bit to ensure it is odd
        c |= 1
        # set highest bit to ensure bit length
        c |= (1 << (bits - 1))

        if FermatPrimalityTest(c):
            return c
            
# extended Euclidean algorithm
def extended_gcd(a, b):
    '''
    Extended Euclidean algorithm to find gcd.

    Parameters: 
    a, b: Input integers.

    Returns: 
    tuple: (gcd, x, y) such that g = gcd(a, b) and (a * x) + (b * y) = g.
    '''
    if b == 0:
        return a, 1, 0
    gcd, x1, y1 = extended_gcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return gcd, x, y

# modular inverse using extended Euclidean algorithm
def mod_inverse(e, phi):
    '''
    Find modular multiplicative inverse of e % phi.

    Parameters:
    e: Number (int) to find inverse for.
    phi: Euler's totient.

    Returns: 
    int or None: Modular inverse if it exists or None.
    '''
    gcd, x, y = extended_gcd(e, phi)
    if gcd != 1:
        return None
    return x % phi

# key generation function
def RSA_key_generation():
    '''
    Generate RSA public and private key pairs.

    Creates:
    p_q.txt: Contains the two primes p and q.
    e_n.txt: Contains public key (e,n).
    d_n.txt: Contains private key (d,n).
        * Note: Each file contains one integer per line with no extra whitespace. *

    Returns:
    None
    '''
    # generate two large prime numbers (128 -  256 bits)
    p = generate_prime(random.randint(128,256))
    q = generate_prime(random.randint(128,256))

    # ensure p and q are different
    while q == p:
        q =  generate_prime(random.randint(128,256))

    # compute n
    n = p*q

    # Euler's totient
    phi = (p - 1) *  (q - 1)

    # choose e
    e = 65537
    if gcd(e, phi) != 1:
        # regenerate primes if e is not coprime with phi
        return  RSA_key_generation()

    # compute d (private exponent)
    d = mod_inverse(e, phi)        
    
    # Open a file in write mode ('w') and save the integer
    with open('p_q.txt', 'w') as file:
        file.write(f"{p}\n")  # save p
        file.write(f"{q}")  # save q

    # save public key (e,  n)
    with open('e_n.txt', 'w') as file:
        file.write(f"{e}\n") # save e
        file.write(f"{n}")  # save n

    # save private key (d, n)
    with open('d_n.txt', 'w') as file:
        file.write(f"{d}\n") # save d
        file.write(f"{n}")  # save n
        
    print("done with key generation!")

In [65]:
def Signing(doc, keyfile = 'd_n.txt'):
    '''
    Digitally sign a document using RSA and SHA-256.

    Parameters: 
    doc: (str) The path to the file to be signed.
    keyfile: (str) File containing the private key (d,n) one integer per line.

    Returns:
    str: The filename of the signed output document.
    '''
    # load private key
    with open(keyfile, 'r') as f:
        d = int(f.readline().strip())
        n = int(f.readline().strip())

    # read document
    with open(doc, 'rb') as f:
        content = f.read()

    # hash the content (32 raw bytes) and convert to integer
    digest_bytes = hashlib.sha256(content).digest()
    h_int = int.from_bytes(digest_bytes, byteorder = 'big')

    # sign: s = h^d % n
    signature = pow(h_int, d, n)

    # convert signature to exactly 64 raw bytes
    sig_len = 64
    try:
        signature_bytes = signature.to_bytes(sig_len, byteorder = 'big')
    except OverflowError:
        raise ValueError('Signature does not fit in 64 bytes. Ensure n < 2^512 (p and q < 256 bits).')

    # write signed file: original content + 64 - byte signature
    out_file = doc + '.signed'
    with open(out_file, 'wb') as f:
        f.write(content)
        f.write(signature_bytes)

    print('signed.')
    return out_file

In [66]:
def verification(doc, keyfile = 'e_n.txt'):
    '''
    Verify a signed document using RSA public key.

    Parameters: 
    doc: (str) The path to the signed document.
    keyfile: (str) Path to the public key file.

    Returns:
    bool: True if signature is valid or False otherwise.
    '''
    # load public key
    with open(keyfile, 'r') as f:
        e = int(f.readline().strip())
        n = int(f.readline().strip())

    # read signed file in binary mode
    with open(doc, 'rb') as f:
        data = f.read()

    # extract the signature (last 64 bytes)
    sig_len = 64
    
    # check if file is long enough to contain a signature
    if len(data)  < sig_len:
        raise ValueError('Signed document too short to contain signature')

    # everything except last 64 bytes
    content = data[:-sig_len]

    # last 64 raw bytes
    signature_bytes = data[-sig_len:]

    # convert signature bytes to integer
    signature = int.from_bytes(signature_bytes, byteorder = 'big')

    # recompute SHA-256 hash of content and convert to integer
    digest_bytes = hashlib.sha256(content).digest()
    h_int = int.from_bytes(digest_bytes, byteorder = 'big')

    # verify: does signature^e % n == hash?
    check = pow(signature, e, n)
    
    if check == h_int:
        print('Authentic!')
        return True
    else:
        print('Modified!')
        return False

In [67]:
def CPSC_435_Project1(part, task="", fileName=""):
    # part I: generate keys, command-line arguments will be: python yourProgram.py 1
    if part == 1:
        RSA_key_generation()
    # part II: signing or verification, command-line will be for example: python yourProgram.py 2 s file.txt
    #                                       or   python yourProgram.py 2 v file.txt.signed
    else:
        if "s" in task:  # do signing
            doc = fileName
            key = 'd_n.txt'
            Signing(doc, key)
        else:
            # do verification
            doc = fileName
            key = 'e_n.txt'
            verification(doc, key)

    print("done!")
    return

In [68]:
# when we grade your part 1 - RSA_key_generation, we call this func
CPSC_435_Project1(1)

done with key generation!
done!


In [69]:
# create a sample document to test signing/verification
with open("contract.txt", "w") as f:
    f.write("This is a test contract for 2a and 2b.")

In [70]:
# when we grade your part 2a - signing,
docName = "contract.txt" # the fileName is just an example
CPSC_435_Project1(2, "s", docName)

signed.
done!


In [71]:
# when we grade your part 2b - verification,
docName = "contract.txt.signed" # the fileName is just an example
CPSC_435_Project1(2, "v", docName)

Authentic!
done!
